# Week 1 — Solutions & Live Exploration (Trainer Only)

Instructor-only companion to `docs/trainer/quiz-answers.md` and `docs/trainer/lab-solutions.md`.
**Do not distribute to learners** — it contains quiz answers and lab solutions.

Every code cell calls the repo's own `radar.*` functions — nothing is reimplemented — so the
answers are *computed*, not quoted, and you can perturb parameters mid-lesson to explore.

Run top-to-bottom. All randomness is seeded via `RadarConfig(seed=42)`.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from radar import signal_gen
from radar.config import RadarConfig

cfg = RadarConfig(seed=42)
print(cfg)

## Quiz 1 — Duty cycle

> With `τ = 20 us` and `T = 1 ms`, what is the duty cycle?

A radar can't listen while it shouts, so it alternates bursts with quiet gaps. Duty cycle is the
fraction of time the transmitter is on. Computed from the config, not from memory:

In [ ]:
duty = cfg.pulse_width_s / cfg.pri_s
print(f"duty cycle = {duty:.1%}")
print(f"quiet (listen) window = {(1 - duty):.1%} of the time")

## Quiz 2 — Round-trip delay (samples)

> A target at `R = 1000 m` — what round-trip delay (in samples at `fs = 20 MHz`)?

The wave travels out *and* back, so the delay is `2R/c`, then converted to samples by multiplying
by `fs`. This is the single most load-bearing number in Week 1 — every later stage indexes arrays by it.

In [ ]:
c = 3e8
t_delay = 2 * cfg.target_range_m / c
n_delay = round(t_delay * cfg.fs_hz)
print(f"round-trip delay = {t_delay*1e6:.2f} us")
print(f"delay in samples = {n_delay}   (expected 133)")

## Quiz 3 — Why the factor of 2?

> Why does `R = c·τ/2` have a factor of 2?

Out and back. The measured delay is the *round trip*; range is half of it. Trivial, but the factor
of 2 is the most common unit bug in radar code — check it every time you convert delay ↔ range.

In [ ]:
R = c * t_delay / 2
assert np.isclose(R, cfg.target_range_m)
print(f"R = c·t_delay/2 = {R:.0f} m  (consistent with the configured target)")

## Quiz 4 — Range resolution

> What is the range resolution of a 5 MHz LFM chirp? Why is that better than a 20 us rectangular pulse?

After pulse compression, **bandwidth** sets resolution, not pulse width: `ΔR = c/(2B)`. The chirp
collects long-pulse energy (`τ·B = 100`) yet resolves like a short pulse.

In [ ]:
delta_r_lfm = c / (2 * cfg.bandwidth_hz)
delta_r_rect = c * cfg.pulse_width_s / 2
print(f"LFM  (B=5 MHz,  τ=20 us): ΔR = {delta_r_lfm:.0f} m")
print(f"Rect (τ=20 us):           ΔR = {delta_r_rect:.0f} m")
print(f"improvement = {delta_r_rect/delta_r_lfm:.0f}x  (time-bandwidth product τ·B = {cfg.bandwidth_hz*cfg.pulse_width_s:.0f})")

## Stage 1 recap — The transmit waveform

Generate the pulse and confirm its two defining properties: length = `τ·fs` samples, and the
instantaneous frequency sweeps `0 → B` across the pulse.

In [ ]:
pulse = signal_gen.lfm_chirp(cfg)
print(f"pulse samples = {pulse.size}  ({pulse.size/cfg.fs_hz*1e6:.1f} us at {cfg.fs_hz/1e6:.0f} MHz)")

t_us = np.arange(pulse.size) / cfg.fs_hz * 1e6
fig, ax = plt.subplots()
ax.plot(t_us, np.real(pulse))
ax.set_xlabel("time (us)")
ax.set_ylabel("Re{s(t)}")
ax.set_title("LFM chirp (real part)")
plt.show()

In [ ]:
phase = np.unwrap(np.angle(pulse))
f_inst = np.diff(phase) / (2 * np.pi) * cfg.fs_hz

fig, ax = plt.subplots()
ax.plot(t_us[1:], f_inst / 1e6)
ax.axhline(cfg.bandwidth_hz / 1e6, color="r", ls="--", lw=1, label="B")
ax.set_xlabel("time (us)")
ax.set_ylabel("instantaneous frequency (MHz)")
ax.set_title(f"sweep {f_inst[0]/1e3:.0f} kHz → {f_inst[-1]/1e6:.2f} MHz (B = {cfg.bandwidth_hz/1e6:.0f} MHz)")
ax.legend()
plt.show()

## Stage 1 recap — The pulse train

64 pulses per CPI, each PRI 20,000 samples long, the pulse in the first 400 samples and silence
after — that silence is where the echo will land (Stage 2).

In [ ]:
train = signal_gen.pulse_train(cfg)
print(f"train shape = {train.shape}  (n_pulses, samples_per_pri)")
print(f"energy per pulse row: {np.sum(np.abs(train) > 0, axis=1).min()} samples (should be 400)")

## Stage 2 — Moving target simulation

*(Scaffold — filled when stage 2 lands.)* `Target`, `propagate`, `simulate_channel`: the echo of
the pulse lands at sample 133 with noise power matching the configured SNR.

## Stage 3 — Matched filter & range estimation

*(Scaffold — filled when stage 3 lands.)* `matched_filter`, `range_from_delay`, `detect_peaks`:
the compressed peak sits at bin ~133, visibly sharper than the raw echo, and the measured range
lands within one range bin of 1000 m at SNR 20 dB.

## Lab L1 — SNR sweep

*(Scaffold — filled when stage 3 lands.)* Sweep SNR 30→0 dB and plot detection error vs SNR.
Expected: failure crossing ~10–15 dB for a fixed threshold — the point being that detection SNR
depends on the threshold policy (`P_fa`), a preview of CFAR.

## Free play

Change **one** parameter, re-run the relevant cell, and explain the effect. Seeded, so results are
reproducible:

- `cfg.pulse_width_s` — what happens to the number of samples per pulse?
- `cfg.pulse_type = "rect"` — how does the waveform (and its spectrum) differ from the chirp?
- `cfg.target_range_m` — where does the echo peak land once stages 2–3 are in place?
- `cfg.fs_hz` — what breaks if you drop the sample rate below `2·B`?